# Binary classification with random forest

In [ ]:
%load_ext autoreload
%autoreload 2

# Re-process dataset into a simpler csv with an explicit 'source' column

this will later be used in the dataset. 

In [ ]:
from pathlib import Path
import pandas as pd
import yaml

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)

for v in Path(config["dataset"]["path"]).glob("*.dat"):
    print(v)
    source = None
    if "AGN" in v.name:
        source = 0
    elif "POPSTAR" in v.name:
        source = 1
    else:
        raise ValueError(f"Unknown source for {v.name}")

    df = pd.read_csv(
        v,
        sep=r"\s+",
        **(config["dataset"]["read_kwargs"] or {}),
        engine=config["dataset"]["engine"],
        comment=config["dataset"]["comment"],
    )

    df["source"] = source

    print(df.head())

    df.to_csv(v.with_suffix(".csv"), index=False, sep=",")

In [ ]:
from GalaxySpectrumClassifier import PandasDataset, to_xy
from torch.utils.data import random_split
import torch
import yaml

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset = PandasDataset.from_config(config["dataset"])
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)
X_train, y_train = to_xy(train_dataset)
X_test, y_test = to_xy(test_dataset)

# Train model 

In [ ]:
from GalaxySpectrumClassifier import SimpleTrainer
import yaml

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)
trainer = SimpleTrainer.from_config(config["trainer"])

trainer.fit(train_dataset)
trainer.save_snapshot("trained_random_forest")

# Test model

In [ ]:
from GalaxySpectrumClassifier import SimpleTrainer

trainer = SimpleTrainer.load_snapshot(
    "../training/binaryclassifier_simple_example/trained_random_forest"
)

In [ ]:
test_results = trainer.test(test_dataset)
test_results = pd.DataFrame.from_dict(
    data=[
        test_results,
    ]
)
test_results.to_csv(
    trainer.output_path / "trained_random_forest/test_results.csv", index=False
)
test_results

that these metrics are so pathologically high is an artifact of the data selection, not really of the quality of the classifier as such